# Plan de accion de Fase 2 (Protocolo experimental)

Este notebook se dedica exclusivamente a cerrar los pendientes de la Fase 2 con trazabilidad reproducible.

Objetivo: convertir la documentacion del protocolo en artefactos ejecutados y exportados (splits, CV, validacion de candidatas y decisiones finales).

## 1. Alcance y entregables

Entregables minimos que deben quedar generados al cerrar esta fase:

1. Split externo estratificado persistido (train_val / test).
2. Configuracion de validacion interna estratificada y semilla comun.
3. Resultados CV por candidata (base vs extendido) para accuracy y recall.
4. Decision final de candidatas (aceptar/rechazar) segun reglas del protocolo.
5. Tabla de trazabilidad final exportada para memoria.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold

BASE_DIR = Path('..')
DATA_PATH = BASE_DIR / 'data' / 'raw' / 'BBDD_ML_TAREA.csv'
TABLES_PATH = BASE_DIR / 'outputs' / 'tables'
TABLES_PATH.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5

## 2. Estado actual automatico

Esta celda inspecciona si ya existen los artefactos clave de Fase 2 y marca su estado actual.

In [2]:
required_artifacts = [
    'fase2_split_indices.csv',
    'fase2_split_config.json',
    'fase2_feature_eval_template.csv',
    'fase2_feature_eval_decisions.csv',
    'fase2_feature_eval_metrics.csv',
    'fase2_action_plan_status.csv'
]

status_rows = []
for f in required_artifacts:
    p = TABLES_PATH / f
    status_rows.append({
        'artifact': f,
        'exists': p.exists(),
        'path': str(p)
    })

artifact_status = pd.DataFrame(status_rows)
display(artifact_status)

,artifact,exists,path
0,fase2_split_indices.csv,False,..\outputs\tables\fase2_split_indices.csv
1,fase2_split_config.json,False,..\outputs\tables\fase2_split_config.json
2,fase2_feature_eval_template.csv,True,..\outputs\tables\fase2_feature_eval_template.csv
3,fase2_feature_eval_decisions.csv,True,..\outputs\tables\fase2_feature_eval_decisions...
4,fase2_feature_eval_metrics.csv,False,..\outputs\tables\fase2_feature_eval_metrics.csv
5,fase2_action_plan_status.csv,False,..\outputs\tables\fase2_action_plan_status.csv


## 3. Plan de accion operativo

Checklist accionable de pendientes. Se actualiza automaticamente con cada ejecucion.

In [3]:
action_plan = pd.DataFrame([
    {'step': 'Crear split externo estratificado y guardarlo', 'owner': 'notebook', 'status': 'pendiente'},
    {'step': 'Guardar configuracion formal del split/CV', 'owner': 'notebook', 'status': 'pendiente'},
    {'step': 'Calcular metricas CV base vs extendido por candidata', 'owner': 'notebook', 'status': 'pendiente'},
    {'step': 'Aplicar regla de decision y exportar aceptar/rechazar', 'owner': 'notebook', 'status': 'pendiente'},
    {'step': 'Generar resumen final para memoria', 'owner': 'notebook', 'status': 'pendiente'}
])

display(action_plan)

,step,owner,status
0,Crear split externo estratificado y guardarlo,notebook,pendiente
1,Guardar configuracion formal del split/CV,notebook,pendiente
2,Calcular metricas CV base vs extendido por can...,notebook,pendiente
3,Aplicar regla de decision y exportar aceptar/r...,notebook,pendiente
4,Generar resumen final para memoria,notebook,pendiente


## 4. Paso 1 - Split externo estratificado

Se construye sobre el conjunto base de modelado (deduplicado y sin variables excluidas por diseno), y se guardan indices para reutilizar exactamente el mismo reparto en todo el proyecto.

In [4]:
df_raw = pd.read_csv(DATA_PATH, na_values=['NA'])
df_work = df_raw.drop_duplicates().copy()
cols_to_drop = ['V4', 'V10', 'V13', 'V16', 'V19']
df_model_base = df_work.drop(columns=cols_to_drop).copy()

X = df_model_base.drop(columns=['Y'])
y = df_model_base['Y']

idx = np.arange(len(df_model_base))
idx_train_val, idx_test = train_test_split(
    idx,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

split_df = pd.DataFrame({
    'row_idx': idx,
    'set': np.where(np.isin(idx, idx_test), 'test', 'train_val')
})
split_df.to_csv(TABLES_PATH / 'fase2_split_indices.csv', index=False)

split_config = {
    'random_state': RANDOM_STATE,
    'test_size': TEST_SIZE,
    'n_splits_cv': N_SPLITS,
    'dataset_shape': list(df_model_base.shape)
}
with open(TABLES_PATH / 'fase2_split_config.json', 'w', encoding='utf-8') as f:
    json.dump(split_config, f, ensure_ascii=False, indent=2)

print('train_val:', (split_df['set'] == 'train_val').sum())
print('test:', (split_df['set'] == 'test').sum())
display(split_df['set'].value_counts())

train_val: 2830
test: 708


set
train_val    2830
test          708
Name: count, dtype: int64

## 5. Paso 2 - Configuracion CV interna

Se registra de forma explicita la configuracion de validacion cruzada estratificada que se usara para evaluar candidatas.

In [5]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

cv_info = pd.DataFrame({
    'parameter': ['n_splits', 'shuffle', 'random_state'],
    'value': [N_SPLITS, True, RANDOM_STATE]
})
display(cv_info)

,parameter,value
0,n_splits,5
1,shuffle,True
2,random_state,42


## 6. Paso 3 y 4 - Pendientes de evaluacion de candidatas

Este notebook deja el plan y la infraestructura listos. El calculo de metricas CV base vs extendido por candidata se implementara en el notebook de evaluacion/modelado para no duplicar logica experimental.

Mientras tanto, se guarda un estado consolidado de avance.

In [6]:
status_map = {
    'Crear split externo estratificado y guardarlo': (TABLES_PATH / 'fase2_split_indices.csv').exists(),
    'Guardar configuracion formal del split/CV': (TABLES_PATH / 'fase2_split_config.json').exists(),
    'Calcular metricas CV base vs extendido por candidata': (TABLES_PATH / 'fase2_feature_eval_metrics.csv').exists(),
    'Aplicar regla de decision y exportar aceptar/rechazar': (TABLES_PATH / 'fase2_feature_eval_decisions.csv').exists(),
    'Generar resumen final para memoria': False
}

action_plan['status'] = action_plan['step'].map(lambda s: 'hecho' if status_map.get(s, False) else 'pendiente')
display(action_plan)
action_plan.to_csv(TABLES_PATH / 'fase2_action_plan_status.csv', index=False)
print('Estado exportado en:', (TABLES_PATH / 'fase2_action_plan_status.csv').resolve())

,step,owner,status
0,Crear split externo estratificado y guardarlo,notebook,hecho
1,Guardar configuracion formal del split/CV,notebook,hecho
2,Calcular metricas CV base vs extendido por can...,notebook,pendiente
3,Aplicar regla de decision y exportar aceptar/r...,notebook,hecho
4,Generar resumen final para memoria,notebook,pendiente


Estado exportado en: D:\Copias de seguridad\2025-2026\Master UCM\Módulo 11\Tarea\outputs\tables\fase2_action_plan_status.csv


## 7. Criterio de salida de Fase 2

La fase se considera cerrada cuando:

- split y configuracion CV esten guardados,
- exista tabla de metricas CV por candidata,
- exista tabla final de decisiones aceptar/rechazar,
- y el estado consolidado marque todo en hecho salvo, como maximo, el resumen narrativo final para memoria.